# Transcript Q&A Demo with NVIDIA
Ask questions about a transcript and get answers with timestamps

## 1. Install Dependencies

In [2]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-nvidia-ai-endpoints
print("✓ Installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.8 MB/s eta 0:00:00
✓ Installed!


## 2. Import Libraries

In [3]:
import os
from typing import List, Dict
from sentence_transformers import SentenceTransformer
import faiss
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage

## 3. Load API Key

In [4]:
from google.colab import userdata
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
print("✓ API key loaded")

✓ API key loaded


## 4. Mock Transcript Data
This simulates what your ASR would produce

In [5]:
# Mock transcript - simulating a tech startup quarterly meeting
segment_timestamps = [
    {'start': 0.0, 'end': 4.5, 'segment': 'Welcome to our Q4 2024 all-hands meeting at TechVision AI.'},
    {'start': 5.0, 'end': 10.2, 'segment': 'Today we will discuss our product roadmap, hiring plans, and financial results.'},
    {'start': 11.0, 'end': 16.8, 'segment': 'Our revenue this quarter reached 2.3 million dollars, which is a 45% increase from last quarter.'},
    {'start': 17.5, 'end': 23.1, 'segment': 'We currently have 47 employees across engineering, sales, and operations teams.'},
    {'start': 24.0, 'end': 29.3, 'segment': 'Our main product is an AI-powered customer service chatbot called ServiceBot.'},
    {'start': 30.0, 'end': 35.7, 'segment': 'ServiceBot has been deployed at 23 enterprise clients including Microsoft and Salesforce.'},
    {'start': 36.5, 'end': 42.1, 'segment': 'We are planning to hire 15 new engineers in the next quarter to expand our ML team.'},
    {'start': 43.0, 'end': 48.9, 'segment': 'Our office is located in San Francisco, but we also have a small team working remotely from Seattle.'},
    {'start': 50.0, 'end': 56.3, 'segment': 'The product roadmap for 2025 includes multi-language support and voice integration features.'},
    {'start': 57.0, 'end': 63.2, 'segment': 'We received Series A funding of 10 million dollars led by Sequoia Capital in September.'},
    {'start': 64.0, 'end': 69.5, 'segment': 'Our customer satisfaction score is currently at 4.7 out of 5 stars based on 150 reviews.'},
    {'start': 70.0, 'end': 75.8, 'segment': 'The engineering team recently launched a new dashboard that reduced response time by 30%.'},
    {'start': 76.5, 'end': 82.1, 'segment': 'We are competing mainly with Intercom and Zendesk in the customer service software market.'},
    {'start': 83.0, 'end': 88.7, 'segment': 'Our marketing budget for next quarter is 500,000 dollars focused on content marketing and SEO.'},
    {'start': 89.5, 'end': 95.2, 'segment': 'The CEO Sarah Chen announced that we are planning an IPO in late 2026.'},
    {'start': 96.0, 'end': 101.8, 'segment': 'Employee benefits include full health insurance, unlimited PTO, and stock options.'},
    {'start': 102.5, 'end': 108.3, 'segment': 'Our biggest challenge right now is scaling our infrastructure to handle 10x more traffic.'},
    {'start': 109.0, 'end': 114.6, 'segment': 'The sales team closed 8 new enterprise deals this quarter worth a total of 1.2 million in ARR.'},
    {'start': 115.5, 'end': 121.2, 'segment': 'We use Python and PyTorch for our machine learning models, deployed on AWS.'},
    {'start': 122.0, 'end': 127.8, 'segment': 'Company culture is very important to us, we have weekly team lunches and monthly offsites.'},
    {'start': 128.5, 'end': 134.1, 'segment': 'Our retention rate is 95%, which means most of our customers stay with us long-term.'},
    {'start': 135.0, 'end': 140.7, 'segment': 'Next month we are attending TechCrunch Disrupt in San Francisco to showcase our product.'},
    {'start': 141.5, 'end': 147.2, 'segment': 'The CTO Michael Park said we will open-source some of our ML tools in early 2025.'},
    {'start': 148.0, 'end': 153.8, 'segment': 'Our pricing model is based on number of conversations handled, starting at 99 dollars per month.'},
    {'start': 154.5, 'end': 160.0, 'segment': 'Thank you everyone for your hard work this quarter, lets make 2025 even better!'}
]

print(f"✓ Loaded {len(segment_timestamps)} transcript segments")
print("\nSample segments:")
for seg in segment_timestamps[:3]:
    print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s]: {seg['segment']}")

✓ Loaded 25 transcript segments

Sample segments:
  [0.0s - 4.5s]: Welcome to our Q4 2024 all-hands meeting at TechVision AI.
  [5.0s - 10.2s]: Today we will discuss our product roadmap, hiring plans, and financial results.
  [11.0s - 16.8s]: Our revenue this quarter reached 2.3 million dollars, which is a 45% increase from last quarter.


## 5. Define Transcript Classes

In [6]:
class TranscriptSegment:
    def __init__(self, text: str, start_time: float, end_time: float):
        self.text = text
        self.start_time = start_time
        self.end_time = end_time

    def format_timestamp(self):
        """Convert seconds to MM:SS format"""
        start_min, start_sec = divmod(int(self.start_time), 60)
        end_min, end_sec = divmod(int(self.end_time), 60)
        return f"{start_min:02d}:{start_sec:02d} - {end_min:02d}:{end_sec:02d}"

def process_timestamps(segment_timestamps: List[Dict]) -> List[TranscriptSegment]:
    """Convert raw timestamps to TranscriptSegment objects"""
    segments = []
    for stamp in segment_timestamps:
        segment = TranscriptSegment(
            text=stamp['segment'],
            start_time=stamp['start'],
            end_time=stamp['end']
        )
        segments.append(segment)
    return segments

# Process the mock data
transcript_segments = process_timestamps(segment_timestamps)
print(f"✓ Processed {len(transcript_segments)} segments")

✓ Processed 25 segments


## 6. Build Vector Database

In [7]:
class TranscriptVectorDB:
    def __init__(self, segments: List[TranscriptSegment]):
        self.segments = segments
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self._build_index()

    def _build_index(self):
        """Create searchable index from transcript"""
        texts = [seg.text for seg in self.segments]
        print("Creating embeddings...")
        self.embeddings = self.embedding_model.encode(texts, show_progress_bar=True)

        # Build FAISS index
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(self.embeddings.astype('float32'))
        print(f"✓ Vector database built with {self.index.ntotal} segments")

    def search(self, query: str, k: int = 5):
        """Find most relevant segments for a query"""
        query_embedding = self.embedding_model.encode([query])
        distances, indices = self.index.search(query_embedding.astype('float32'), k)

        results = []
        for idx, distance in zip(indices[0], distances[0]):
            results.append((self.segments[idx], float(distance)))
        return results

# Build the vector database
vector_db = TranscriptVectorDB(transcript_segments)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Vector database built with 25 segments


## 7. Initialize NVIDIA LLM

In [8]:
llm = ChatNVIDIA(
    model="meta/llama-3.1-8b-instruct",
    temperature=0.2,
    max_tokens=500
)

/tmp/ipython-input-3035775012.py:1: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


## 8. Create Q&A System

In [9]:
class TranscriptQA:
    def __init__(self, vector_db, llm):
        self.vector_db = vector_db
        self.llm = llm

    def answer_question(self, question: str, top_k: int = 5, show_sources: bool = True):
        """Answer questions about the transcript"""
        # Find relevant segments
        relevant_segments = self.vector_db.search(question, k=top_k)

        # Build context
        context_parts = []
        for i, (segment, score) in enumerate(relevant_segments):
            context_parts.append(
                f"Segment {i+1} [{segment.format_timestamp()}]:\n{segment.text}"
            )

        context = "\n\n".join(context_parts)

        # Create prompt
        system_prompt = """You are a helpful assistant that answers questions based on transcript segments.
        Always reference the timestamp(s) when providing answers. Be concise and accurate.
        If the answer is not in the provided segments, say so."""

        user_prompt = f"""Based on the following transcript segments, answer this question: {question}

Transcript segments:
{context}

Please provide a clear answer and reference the relevant timestamps."""

        # Get answer from LLM
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]

        response = self.llm.invoke(messages)
        answer = response.content

        # Show sources
        if show_sources:
            sources = "\n\n📍 Source Segments:\n"
            for i, (segment, score) in enumerate(relevant_segments):
                sources += f"\n{i+1}. [{segment.format_timestamp()}] {segment.text}"
            answer += sources

        return answer

# Initialize Q&A system
qa_system = TranscriptQA(vector_db, llm)
print("✓ Q&A system ready!")

✓ Q&A system ready!


## 9. Ask Questions!
Now you can ask questions about the transcript

In [10]:
# Example questions
questions = [
    "What is the company's revenue?",
    "How many employees does the company have?",
    "What is the main product?",
    "Where is the office located?",
    "Who are the main competitors?"
]

print("="*80)
print("DEMO: ASKING QUESTIONS ABOUT THE TRANSCRIPT")
print("="*80)

for question in questions:
    print(f"\n{'='*80}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*80}")

    answer = qa_system.answer_question(question, top_k=3, show_sources=True)
    print(f"\n💡 ANSWER:\n{answer}")
    print()

DEMO: ASKING QUESTIONS ABOUT THE TRANSCRIPT

❓ QUESTION: What is the company's revenue?

💡 ANSWER:
The company's revenue this quarter is 2.3 million dollars. 

This information can be found in Segment 1 [00:11 - 00:16].

📍 Source Segments:

1. [00:11 - 00:16] Our revenue this quarter reached 2.3 million dollars, which is a 45% increase from last quarter.
2. [01:23 - 01:28] Our marketing budget for next quarter is 500,000 dollars focused on content marketing and SEO.
3. [01:49 - 01:54] The sales team closed 8 new enterprise deals this quarter worth a total of 1.2 million in ARR.


❓ QUESTION: How many employees does the company have?

💡 ANSWER:
The company has 47 employees. 

Reference: Segment 1 [00:17 - 00:23].

📍 Source Segments:

1. [00:17 - 00:23] We currently have 47 employees across engineering, sales, and operations teams.
2. [01:36 - 01:41] Employee benefits include full health insurance, unlimited PTO, and stock options.
3. [00:30 - 00:35] ServiceBot has been deployed at 23 en

## 10. Ask Your Own Questions

In [11]:
# Ask a custom question
my_question = "Who is their CEO"  # Change this!

print(f"❓ YOUR QUESTION: {my_question}")
print("="*80)

answer = qa_system.answer_question(my_question, top_k=5, show_sources=True)
print(f"\n💡 ANSWER:\n{answer}")

❓ YOUR QUESTION: Who is their CEO

💡 ANSWER:
The CEO is Sarah Chen. 

[01:29 - 01:35]: The CEO Sarah Chen announced that we are planning an IPO in late 2026.

📍 Source Segments:

1. [01:29 - 01:35] The CEO Sarah Chen announced that we are planning an IPO in late 2026.
2. [00:17 - 00:23] We currently have 47 employees across engineering, sales, and operations teams.
3. [01:36 - 01:41] Employee benefits include full health insurance, unlimited PTO, and stock options.
4. [02:21 - 02:27] The CTO Michael Park said we will open-source some of our ML tools in early 2025.
5. [00:57 - 01:03] We received Series A funding of 10 million dollars led by Sequoia Capital in September.


## 11. Interactive Q&A

In [12]:
# Interactive mode - ask multiple questions
print("Type 'quit' to exit\n")

while True:
    user_question = input("\n❓ Your question: ")

    if user_question.lower() in ['quit', 'exit', 'q']:
        print("👋 Goodbye!")
        break

    if not user_question.strip():
        continue

    print("\n🤔 Thinking...\n")
    answer = qa_system.answer_question(user_question, top_k=5, show_sources=True)
    print(f"💡 ANSWER:\n{answer}")
    print("\n" + "="*80)

Type 'quit' to exit


❓ Your question: How many clients do they have ?

🤔 Thinking...

💡 ANSWER:
They have 23 enterprise clients. 

Timestamp: [00:30 - 00:35]

📍 Source Segments:

1. [00:17 - 00:23] We currently have 47 employees across engineering, sales, and operations teams.
2. [01:16 - 01:22] We are competing mainly with Intercom and Zendesk in the customer service software market.
3. [00:30 - 00:35] ServiceBot has been deployed at 23 enterprise clients including Microsoft and Salesforce.
4. [02:28 - 02:33] Our pricing model is based on number of conversations handled, starting at 99 dollars per month.
5. [00:36 - 00:42] We are planning to hire 15 new engineers in the next quarter to expand our ML team.


❓ Your question: Whats in the roadmap for 2025 ?

🤔 Thinking...

💡 ANSWER:
According to Segment 2 [00:50 - 00:56], the product roadmap for 2025 includes multi-language support and voice integration features.

📍 Source Segments:

1. [02:34 - 02:40] Thank you everyone for your hard 